# Custom Mask Inference Demo
This notebook demonstrates how to feed a custom risk-mapped mask into the FNOSDF and PNO pipeline to generate a cost-to-go map, using actual goals and ground truth from the dataset.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

from models.fno import FNO2d
from models.deepnormMultiGoal import DEEPNORM2dMultiGoal

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 1. Load Data and Create Custom Mask
We load the distance map, goal, and ground truth output from the city dataset.

In [2]:
data_dir = "dataset/synthetic/64x64/"
dist_maps = np.load(data_dir + "dist_in.npy")
goal_maps = np.load(data_dir + "goal.npy")
output_maps = np.load(data_dir + "output.npy")
actual_masks = np.load(data_dir + "mask.npy")
####
risk_maps    = np.load(data_dir + "risk_maps.npy")

# Choose map index 6
map_idx = 0
dist_map = dist_maps[map_idx]
goal_data = goal_maps[map_idx][::-1] # Swap to [row, col] for correct indexing
gt_output = output_maps[map_idx]
actual_mask = actual_masks[map_idx]
####
risk_map    = risk_maps[map_idx] 

print(f"Map Index: {map_idx}")
print(f"Goal Coordinates: {goal_data}")
print(f"Risk map range: [{risk_map.min():.3f}, {risk_map.max():.3f}]")

Map Index: 0
Goal Coordinates: [ 6 23]
Risk map range: [0.000, 1.000]


## 2. Load Pretrained Models

In [3]:
def smooth_chi(mask, dist, smooth_coef=5.0):
    return torch.mul(torch.tanh(dist * smooth_coef), (mask - 0.5)) + 0.5


# 1. Load FNOSDF
modelSDF = FNO2d(4, 1, 8, 8, 16).to(device)
try:
    modelSDF.load_state_dict(torch.load("./results/FNOSDF/best_model.pt", map_location=device, weights_only=True))
    print("Loaded FNOSDF model weights.")
except FileNotFoundError:
    print("Error: FNOSDF weights not found.")
modelSDF.eval()

# 2. Load PNOwPINN
modelPNOwPINN = DEEPNORM2dMultiGoal(4, 8, 8, 16, in_channels=2).to(device)
try:
    modelPNOwPINN.load_state_dict(torch.load("./results/PNOwPINN/best_model.pt", map_location=device, weights_only=True))
    print("Loaded PNO model weights.")
except FileNotFoundError:
    print("Error: PNO weights not found.")
modelPNOwPINN.eval()

Error: FNOSDF weights not found.


TypeError: DEEPNORM2dMultiGoal.__init__() got an unexpected keyword argument 'in_channels'

## 3. Inference Pipeline
We process all layer masks, average their SDFs and masks, and run both PNO models using the dataset goal.

In [ ]:
# cat(chi, risk) [B,N,N,2] → model

goal_coord = torch.tensor([goal_data], dtype=torch.int).to(device)

mask_tensor = torch.tensor(actual_mask, dtype=torch.float).reshape(1, actual_mask.shape[0], actual_mask.shape[1], 1).to(device)
risk_tensor = torch.tensor(risk_map,    dtype=torch.float).reshape(1, risk_map.shape[0],    risk_map.shape[1],    1).to(device)

with torch.no_grad():
    # Step 1: FNO_SDF predicts d_S(x)
    sdf = modelSDF(mask_tensor)                         # [1,N,N,1]

    # Step 2: Construct chi
    chi = smooth_chi(mask_tensor, sdf, 5.0)             # [1,N,N,1]

    # Step 3: Cancat risk
    chi_with_risk = torch.cat([chi, risk_tensor], dim=-1)  # [1,N,N,2]

    # Step 4: PNO inference
    cost_to_go = modelPNOwPINN(chi_with_risk, goal_coord)  # [1,N,N,1]

cost_to_go_masked = cost_to_go * mask_tensor

print("Inference complete.")
print(f"Output shape: {cost_to_go_masked.shape}")

## 4. Visualization

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(30, 5))

# 1. Original mask
axes[0].imshow(actual_mask, origin="lower", cmap="gray")
axes[0].set_title("Actual Mask")
axes[0].axis("off")

# 2. Risk map
im0 = axes[1].imshow(risk_map, origin="lower", cmap="inferno_r")
axes[1].set_title("Continuous Risk Map")
fig.colorbar(im0, ax=axes[1], label="Risk")
axes[1].axis("off")

# 3. Predicted SDF
im1 = axes[2].imshow(sdf[0].cpu().numpy().squeeze(), origin="lower", cmap="inferno_r")
axes[2].set_title("Predicted SDF")
fig.colorbar(im1, ax=axes[2], label="Distance")
axes[2].axis("off")

# 4. Predicted cost-to-go
im2 = axes[3].imshow(cost_to_go_masked[0].cpu().numpy().squeeze(), origin="lower", cmap="inferno_r")
axes[3].plot(goal_data[1], goal_data[0], "ro", markersize=5)
axes[3].set_title("Predicted Cost-to-go\n(PNO w/ PINN + Continuous Risk)")
fig.colorbar(im2, ax=axes[3], label="Cost")
axes[3].axis("off")

# 5. Ground truth
im3 = axes[4].imshow(gt_output.squeeze(), origin="lower", cmap="inferno_r")
axes[4].plot(goal_data[1], goal_data[0], "ro", markersize=5)
axes[4].set_title("Ground Truth (FMM)")
fig.colorbar(im3, ax=axes[4], label="Cost")
axes[4].axis("off")

plt.suptitle(f"Continuous Risk Map Approach — Map {map_idx}", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Path Planning with A*
We use the predicted cost-to-go map from PNO w/ PINN as a heuristic for A* planning. We'll pick a start position far from the goal and plan the path.

In [ ]:
import sys
import os
import pqdict
# Add src to the path to import the astar module
heuristics_dir = os.path.abspath("../src")
if heuristics_dir not in sys.path:
    sys.path.append(heuristics_dir)

from astar.astar import AStar
from astar.environment_simple import Environment2D
from astar.utilities import drawMap, drawPath2D

# 1. Prepare Environment
# cmap: 0 for free space, 1 for obstacle
cmap = 1.0 - actual_mask

# Heuristic: the predicted cost-to-go map from PNO w/ PINN
heuristic_map = cost_to_go_masked[0].cpu().numpy().squeeze()

# Create environment
env = Environment2D(goal_data, cmap, valuefunction=heuristic_map)

# 2. Select a Start Position
# Find all passable indices
passable_indices = np.argwhere(cmap == 0)
# Pick a point with a high cost-to-go as the start position to ensure a long path
valid_costs = [(idx, heuristic_map[idx[0], idx[1]]) for idx in passable_indices]
valid_costs.sort(key=lambda x: x[1], reverse=True)
start_coord = valid_costs[0][0]

print(f"Start Coordinates: {start_coord}")
print(f"Goal Coordinates: {goal_data}")

# 3. Run A* Planning
path_cost, path, action_idx, nodes_count, sss = AStar.plan(start_coord, env)

print(f"Path Cost: {path_cost}")
print(f"Number of Node Expansions: {nodes_count}")

path_array = np.asarray(path)

In [ ]:
# 4. Visualize the path
fig, ax = plt.subplots(figsize=(6, 4))
drawMap(ax, cmap)

if len(path_array) > 0:
    drawPath2D(ax, path_array)

ax.set_title("A* Path Planning with PNO Heuristic")
plt.show()